# Test Set Evaluation — ESM3ΔG & SaProtΔG
Loads `test_set_evaluation_results.json` and computes RMSE, Spearman, and diagnostic plots.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr, pearsonr

RESULTS_PATH = "test_set_evaluation_results.json"
DG_MIN, DG_MAX = -1.0, 5.0

## Load results

In [ ]:
with open(RESULTS_PATH) as f:
    data = json.load(f)

records = data["records"]

sap_rows = [r for r in records if r["sap_mean"] is not None]
esm_rows = [r for r in records if r["esm_mean"] is not None]

print(f"Total records : {len(records)}")
print(f"SaProtΔG valid: {len(sap_rows)}")
print(f"ESM3ΔG   valid: {len(esm_rows)}")

## Compute metrics

In [ ]:
def metrics(rows, pred_key, true_key="dg_true"):
    y_true = np.array([r[true_key] for r in rows])
    y_pred = np.array([r[pred_key] for r in rows])
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    sp   = float(spearmanr(y_true, y_pred).statistic)
    pe   = float(pearsonr(y_true, y_pred).statistic)
    return dict(n=len(rows), rmse=rmse, mae=mae, spearman=sp, pearson=pe), y_true, y_pred

results_table = []

if sap_rows:
    sap_m, sap_true, sap_pred = metrics(sap_rows, "sap_mean")
    results_table.append({"Model": "SaProtΔG (ensemble)", **sap_m})

if esm_rows:
    esm_m, esm_true, esm_pred = metrics(esm_rows, "esm_mean")
    results_table.append({"Model": "ESM3ΔG (ensemble)", **esm_m})

df_metrics = pd.DataFrame(results_table).set_index("Model")
df_metrics = df_metrics.rename(columns={"n": "N", "rmse": "RMSE", "mae": "MAE",
                                         "spearman": "Spearman ρ", "pearson": "Pearson r"})
df_metrics.round(4)

## Plots

In [ ]:
def scatter_panel(ax, y_true, y_pred, label, color, m):
    ax.scatter(y_true, y_pred, s=4, alpha=0.3, color=color, rasterized=True)
    lim = [DG_MIN - 0.2, DG_MAX + 0.2]
    ax.plot(lim, lim, "k--", lw=0.8, alpha=0.5)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Experimental ΔG (kcal/mol)", fontsize=11)
    ax.set_ylabel("Predicted ΔG (kcal/mol)", fontsize=11)
    ax.set_title(label, fontsize=13, fontweight="bold")
    info = f"N={m['N']}\nRMSE={m['RMSE']:.3f}\nSpearman={m['Spearman ρ']:.3f}"
    ax.text(0.04, 0.96, info, transform=ax.transAxes, fontsize=9,
            va="top", bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

n_panels = int(bool(sap_rows)) + int(bool(esm_rows))
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5), squeeze=False)
axes = axes[0]

panel = 0
if sap_rows:
    scatter_panel(axes[panel], sap_true, sap_pred, "SaProtΔG", "#2196F3",
                  df_metrics.loc["SaProtΔG (ensemble)"])
    panel += 1
if esm_rows:
    scatter_panel(axes[panel], esm_true, esm_pred, "ESM3ΔG", "#4CAF50",
                  df_metrics.loc["ESM3ΔG (ensemble)"])

plt.suptitle("DMSv4-AF Test Set — Predicted vs Experimental ΔG", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("scatter_pred_vs_true.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Error distribution
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 4), squeeze=False)
axes = axes[0]

panel = 0
if sap_rows:
    err = sap_pred - sap_true
    axes[panel].hist(err, bins=60, color="#2196F3", alpha=0.7, edgecolor="white", lw=0.3)
    axes[panel].axvline(0, color="black", lw=1, ls="--")
    axes[panel].set_xlabel("Prediction error (kcal/mol)", fontsize=11)
    axes[panel].set_ylabel("Count", fontsize=11)
    axes[panel].set_title("SaProtΔG error distribution", fontsize=13, fontweight="bold")
    axes[panel].text(0.97, 0.96, f"MAE={sap_m['mae']:.3f}", transform=axes[panel].transAxes,
                     ha="right", va="top", fontsize=9,
                     bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))
    panel += 1

if esm_rows:
    err = esm_pred - esm_true
    axes[panel].hist(err, bins=60, color="#4CAF50", alpha=0.7, edgecolor="white", lw=0.3)
    axes[panel].axvline(0, color="black", lw=1, ls="--")
    axes[panel].set_xlabel("Prediction error (kcal/mol)", fontsize=11)
    axes[panel].set_ylabel("Count", fontsize=11)
    axes[panel].set_title("ESM3ΔG error distribution", fontsize=13, fontweight="bold")
    axes[panel].text(0.97, 0.96, f"MAE={esm_m['mae']:.3f}", transform=axes[panel].transAxes,
                     ha="right", va="top", fontsize=9,
                     bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

plt.suptitle("DMSv4-AF Test Set — Prediction Error Distribution", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("error_distribution.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Ensemble agreement — individual model predictions vs mean (SaProt only)
if sap_rows and all(len(r["sap_preds"]) == 3 for r in sap_rows):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    colors = ["#1565C0", "#0288D1", "#00ACC1"]
    for i, ax in enumerate(axes):
        y_i    = np.array([r["sap_preds"][i] for r in sap_rows])
        y_mean = np.array([r["sap_mean"]      for r in sap_rows])
        sp_i   = spearmanr(sap_true, y_i).statistic
        ax.scatter(sap_true, y_i, s=4, alpha=0.25, color=colors[i], rasterized=True)
        lim = [DG_MIN - 0.2, DG_MAX + 0.2]
        ax.plot(lim, lim, "k--", lw=0.8, alpha=0.5)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("Experimental ΔG", fontsize=10)
        ax.set_ylabel("Predicted ΔG", fontsize=10)
        ax.set_title(f"SaProtΔG model {i+1}  ρ={sp_i:.3f}", fontsize=11, fontweight="bold")
    plt.suptitle("SaProtΔG — Individual Ensemble Members", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig("ensemble_members_saprot.pdf", bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# ΔG bin-level Spearman (how well does ranking hold across the stability range?)
if sap_rows:
    bins = np.linspace(DG_MIN, DG_MAX, 13)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_sp = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (sap_true >= lo) & (sap_true < hi)
        if mask.sum() >= 10:
            bin_sp.append(spearmanr(sap_true[mask], sap_pred[mask]).statistic)
        else:
            bin_sp.append(np.nan)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(bin_centers, bin_sp, width=(bins[1]-bins[0])*0.85,
           color="#2196F3", alpha=0.7, edgecolor="white")
    ax.axhline(sap_m["spearman"], color="black", lw=1.2, ls="--",
               label=f"Overall ρ = {sap_m['spearman']:.3f}")
    ax.set_xlabel("Experimental ΔG bin (kcal/mol)", fontsize=11)
    ax.set_ylabel("Spearman ρ within bin", fontsize=11)
    ax.set_title("SaProtΔG — Per-bin Spearman", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_ylim(-1, 1)
    plt.tight_layout()
    plt.savefig("perbin_spearman_saprot.pdf", bbox_inches="tight", dpi=150)
    plt.show()